# Stage 6: Semantic Cluster Naming with Anime Image Taggers

## Overview and Purpose
A central architectural principle of this repository is that **visual clustering must precede semantic naming**. Anime image taggers (such as WD14 or PixAI) can fail on rare characters, obscure fan-art, alternate outfits, or original characters (OCs) that were never part of the model's Danbooru training tags. By clustering on visual CCIP embeddings first, images are reliably grouped by visual identity regardless of tagger availability.

Once visual clusters exist, **Stage 6** optionally assigns semantic names:
1. **Representative Sampling**: Selects up to $N$ crops per cluster closest to the cluster centroid.
2. **Tagger Inference**: Queries specialized anime taggers (`pixai` with `wd14` fallback) to predict character tags.
3. **Cluster-Level Weighted Voting**: Aggregates predicted character tags across all sampled cluster images.
4. **Decision Acceptance Rules**: A suggested name is accepted only if it satisfies three strict criteria:
   - **Minimum Tag Share**: $\text{share} = \frac{\text{count}(T)}{N} \ge \text{min\_name\_share}$ (default: 30%)
   - **Weighted Score**: $\text{score}_{w}(T) \ge \text{min\_weighted\_score}$ (default: 0.50)
   - **Confidence Margin**: $\text{margin} = \text{score}_{w}(T) - \text{score}_{w}(T_2) \ge \text{min\_top\_margin}$ (default: 0.05)
5. **Folder Rename Plan**: Generates an auditable rename plan (`folder_rename_plan.csv`), leaving uncertain clusters as `cluster_XXXXX_unknown` rather than applying incorrect character names.

## 1. Imports and Environment Setup
Import tagger routines, aggregation voting, and renaming logic.

In [ ]:
from pathlib import Path
import os
import pandas as pd
from IPython.display import display

from anime_character_organizer import (
    TaggingConfig,
    run_cluster_naming,
)
from anime_character_organizer.tagging.aggregation import aggregate_cluster_tags
from anime_character_organizer.tagging.renaming import build_folder_rename_plan
from anime_character_organizer.utils.runs import find_latest_run

## 2. Configuration & Decision Thresholds
Configure tagger model families, voting thresholds, and rename output options.

In [ ]:
PROJECT_DIR = Path("./anime_character_pipeline").expanduser().resolve()
PREVIOUS_CLUSTERING_RUN = None  # Auto-discovers latest 04_hdbscan_clustering run

# Tagger selection: 'pixai' or 'wd14'
PRIMARY_TAGGER = "pixai"
PIXAI_MODEL_NAME = "v0.9"
WD14_MODEL_NAME = "SwinV2_v3"
USE_WD14_FALLBACK = True

# Sampling parameters
MIN_CLUSTER_SIZE_TO_NAME = 2
MAX_IMAGES_PER_CLUSTER = 12

# Decision thresholds for accepting a character name:
MIN_CHARACTER_SCORE = 0.65       # Minimum model confidence score
MIN_NAME_SHARE = 0.30            # Minimum fraction of cluster crops agreeing
MIN_WEIGHTED_SCORE = 0.50        # Minimum weighted score
MIN_TOP_MARGIN = 0.05            # Minimum margin over second-best candidate tag

# Output generation
CREATE_NAMED_OUTPUT = True       # Materializes an additional named folder tree in organized_output/

print("Semantic Naming Configuration:")
print(f"  Primary Tagger: {PRIMARY_TAGGER} (fallback: {USE_WD14_FALLBACK})")
print(f"  Voting Thresholds: share>={MIN_NAME_SHARE}, weighted_score>={MIN_WEIGHTED_SCORE}, margin>={MIN_TOP_MARGIN}")
print(f"  Create Named Output Tree: {CREATE_NAMED_OUTPUT}")

## 3. Semantic Cluster Naming Workflow Execution
Run tagger inference across representative cluster samples, aggregate character votes, and build the rename plan.

In [ ]:
results = run_cluster_naming(
    project_dir=PROJECT_DIR,
    previous_clustering_run_dir=PREVIOUS_CLUSTERING_RUN,
    primary_tagger=PRIMARY_TAGGER,
    pixai_model_name=PIXAI_MODEL_NAME,
    wd14_model_name=WD14_MODEL_NAME,
    use_wd14_fallback=USE_WD14_FALLBACK,
    min_cluster_size_to_name=MIN_CLUSTER_SIZE_TO_NAME,
    max_images_per_cluster_to_tag=MAX_IMAGES_PER_CLUSTER,
    min_character_score=MIN_CHARACTER_SCORE,
    min_name_share=MIN_NAME_SHARE,
    min_weighted_score=MIN_WEIGHTED_SCORE,
    min_top_margin=MIN_TOP_MARGIN,
    create_named_output=CREATE_NAMED_OUTPUT,
    show_progress=True,
)

print(f"Semantic cluster naming completed.")
print(f"Run directory: {results['run_dir']}")

## 4. Accepted Semantic Names Summary
Review clusters where tagger voting met all confidence thresholds and successfully assigned a character name.

In [ ]:
suggestions_df = results["cluster_name_suggestions_df"]
accepted_df = suggestions_df[suggestions_df["accepted_name"]].copy()

print(f"Accepted Character Names: {len(accepted_df)} / {len(suggestions_df)} candidate clusters")
display(accepted_df[[
    "cluster_label", "cluster_folder_stub", "suggested_character_tag",
    "proposed_named_folder", "best_tag_share", "best_weighted_score", "top_margin"
]].head(30))

## 5. Uncertain and Unnamed Clusters
Clusters that do not pass agreement or margin thresholds retain their generic folder stubs (`cluster_XXXXX_unknown`). Reviewing the top tags provides insight into candidate names for manual confirmation.

In [ ]:
uncertain_df = suggestions_df[~suggestions_df["accepted_name"]].copy()
print(f"Uncertain Clusters ({len(uncertain_df)} clusters retained with generic stubs):")
display(uncertain_df[[
    "cluster_label", "cluster_folder_stub", "best_raw_tag",
    "best_tag_share", "best_weighted_score", "top_margin"
]].head(30))

## 6. Folder Rename Plan Preview
Preview how the final folder structure maps original visual cluster stubs to semantic character folder names.

In [ ]:
preview_df = results["folder_rename_preview_df"]
print("Folder Rename Plan Preview:")
display(preview_df.head(40))

## 7. Assumptions, Limitations, and Final Conclusions

### Assumptions & Limitations
- **Danbooru Tag Vocabulary**: PixAI and WD14 models predict tags within their fixed training vocabulary. Original characters (OCs) or non-indexed series will naturally not receive high-confidence character tags and are safely retained as generic clusters.
- **Tag Disambiguation**: Characters with common series aliases or multiple versions (e.g. Saber / Artoria Pendragon) may split votes among aliases. The margin threshold ensures only distinct, dominant names are accepted automatically.

### Pipeline Wrap-Up
This completes the six-stage automatic anime character image organization pipeline. The resulting organized folder tree (under `organized_output/`) provides a clean, validated, non-destructive organization of character identities.